In [ ]:
import scanpy as sc
import scvi
import pandas as pd
from pathlib import Path
import os

# --- Configuration & Paths ---
# Set input/output directories
input_dir = Path("/mnt/d/WY_single_cell_analysis/data/proceed/qc_filtered")
output_dir = Path("/mnt/d/WY_single_cell_analysis/data/proceed/integrated")
output_dir.mkdir(parents=True, exist_ok=True)

# --- 1. Data Loading & Concatenation ---
print("Step 1: Merging filtered h5ad files...")
# Use glob to find all filtered samples
h5ad_files = sorted(list(input_dir.glob("qc_filtered_neo_*.h5ad")))
adatas = [sc.read_h5ad(f) for f in h5ad_files]

# Merge into a single AnnData object using 'inner' join to keep common genes
adata_combined = sc.concat(adatas, join="inner", label="batch")
adata_combined.obs_names_make_unique() # Ensure unique barcodes for each cell

print(f"Total Cells: {adata_combined.n_obs}, Common Genes: {adata_combined.n_vars}")

# --- 2. Feature Selection (HVG) ---
# Store raw counts in a layer for scVI training (scVI requires integer counts)
adata_combined.layers["counts"] = adata_combined.X.copy()

# Normalization for HVG selection (does not affect raw counts in 'counts' layer)
sc.pp.normalize_total(adata_combined, target_sum=1e4)
sc.pp.log1p(adata_combined)

print("Step 2: Selecting Highly Variable Genes (HVG)...")
# flavor="seurat_v3" is recommended for count data. 
# batch_key helps find genes that are variable within each batch.
sc.pp.highly_variable_genes(
    adata_combined, 
    n_top_genes=3000, 
    subset=False, 
    layer="counts", 
    flavor="seurat_v3", 
    batch_key="batch"
)

# --- 3. scVI Integration (Batch Correction) ---
print("Step 3: Training scVI model for batch effect correction...")
# Initialize scVI with 'batch' as the categorical covariate to remove
scvi.model.SCVI.setup_anndata(adata_combined, layer="counts", batch_key="batch")

# Define and train the Variational Autoencoder (VAE)
vae_integrated = scvi.model.SCVI(adata_combined, n_layers=2, n_latent=30, gene_likelihood="nb")
vae_integrated.train()

# Extract the latent representation (low-dimensional integration coordinates)
adata_combined.obsm["X_scVI"] = vae_integrated.get_latent_representation()

# --- 4. Dimensionality Reduction & Visualization ---
print("Step 4: Running Neighbors and UMAP...")
sc.pp.neighbors(adata_combined, use_rep="X_scVI")
sc.tl.umap(adata_combined)

# --- 5. Data Export & Saving ---
final_h5ad_path = output_dir / "adata_combined_integrated.h5ad"
model_save_path = output_dir / "scvi_integrated_model"

# Save the integrated AnnData object
adata_combined.write(final_h5ad_path)
# Save the trained model for future use (e.g., cell type mapping)
vae_integrated.save(model_save_path, overwrite=True)

print(f"Success! Integrated data saved to: {final_h5ad_path}")

# --- 6. Plotting QC Markers ---
sc.pl.umap(
    adata_combined, 
    color=['batch', 'group', 'pct_counts_zikv'], 
    wspace=0.4, 
    show=True,
    title=['Batch Effect', 'Experimental Group', 'ZIKV Load (%)']
)